# 03 - Séparation Train / Validation / Test

On part de `comptes_rendus_variables_cibles.csv` (sortie du notebook 02, avec `SentimentLabel` et
`ChurnRisk` ) et on construit un split **Train ~80% / Validation ~10% / Test ~10%**

## Pourquoi pas un split aléatoire classique ?

 Un split aléatoire classique risquerait de placer deux
variantes du **même template** de part et d'autre de la frontière Train/Test : le modèle aurait alors
"vu" la structure de la phrase pendant l'entraînement et son score en test serait artificiellement gonflé
(data leakage).

## Solution retenue

1. **`template_id`** : on normalise chaque texte (nombres remplacés par un marqueur générique) pour ne
   garder que sa structure, et on regroupe les textes qui partagent la même structure normalisée sous un
   même `template_id`.
2. **`GroupShuffleSplit` par `template_id`** : un même template ne peut jamais se retrouver dans deux
   splits différents.


### 1. Chargement des données (sortie du notebook 02)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import ast

chemin_fichier = Path("/content/drive/MyDrive/data/processed/comptes_rendus_variables_cibles.csv")
df = pd.read_csv(chemin_fichier)

if "Tokens" in df.columns and df["Tokens"].dtype == object:
    df["Tokens"] = df["Tokens"].apply(ast.literal_eval)

print("Corpus chargé :", chemin_fichier)
print("Dimensions :", df.shape)


Mounted at /content/drive
Corpus chargé : /content/drive/MyDrive/data/processed/comptes_rendus_variables_cibles.csv
Dimensions : (1000, 23)


### 2. Configuration

In [2]:
CONFIG = {
    "TEXT_COL": "Compte_Rendu_Text",
    "TRAIN_SIZE": 0.80,
    "VAL_SIZE": 0.10,   # sur le total (donc 50% du reste après extraction du train)
    "TEST_SIZE": 0.10,  # sur le total
    "RANDOM_STATE": 42,
}

assert abs(CONFIG["TRAIN_SIZE"] + CONFIG["VAL_SIZE"] + CONFIG["TEST_SIZE"] - 1.0) < 1e-9, \
    "TRAIN_SIZE + VAL_SIZE + TEST_SIZE doit faire 1.0"
assert CONFIG["TEXT_COL"] in df.columns, f"{CONFIG['TEXT_COL']} absent du DataFrame"


### 3.1 Création de `template_id`

**Objectif** : éviter les doublons et le data leakage. Des textes ayant la même structure (mêmes mots,
seuls les nombres/montants changent) doivent partager le même `template_id` — les variations (montant,
agence, nom...) ne doivent pas créer un nouveau template.


In [3]:
import re


def normaliser_template(texte):
    '''Neutralise ce qui varie d'un client à l'autre pour ne garder que la
    structure de la phrase : les nombres (montants, dates, identifiants) sont
    remplacés par un marqueur générique, la casse et les espaces multiples
    sont normalisés.'''
    t = str(texte).lower()
    t = re.sub(r"\d+([.,]\d+)?", "<NUM>", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t


df["template_norm"] = df[CONFIG["TEXT_COL"]].apply(normaliser_template)
df["template_id"] = df["template_norm"].astype("category").cat.codes

n_templates = df["template_id"].nunique()
print(f"{n_templates} templates uniques pour {len(df)} lignes "
      f"(moyenne de {len(df) / n_templates:.1f} occurrences par template)")
print()
print("Distribution du nombre d'occurrences par template :")
print(df["template_id"].value_counts().describe())


22 templates uniques pour 1000 lignes (moyenne de 45.5 occurrences par template)

Distribution du nombre d'occurrences par template :
count    22.000000
mean     45.454545
std       8.770770
min      29.000000
25%      39.750000
50%      46.000000
75%      49.750000
max      63.000000
Name: count, dtype: float64


In [4]:
# Contrôle visuel : les templates les plus fréquents
top_templates = df["template_id"].value_counts().head(5)
for tid, n in top_templates.items():
    exemple = df.loc[df["template_id"] == tid, CONFIG["TEXT_COL"]].iloc[0]
    print(f"template_id={tid}  (n={n})")
    print(f"  ex: {exemple}")
    print()


template_id=8  (n=63)
  ex: Frais barcha 3al compte courant w ena ma nesta3mloch dima. Ken bech tkamlo haka bech nbadal el banka o5ra.

template_id=16  (n=57)
  ex: الحريف جا للفرع غاضب على خاطر تقصتلو خطية متع شيك بدون رصيد. يطالب يقابل الشيف دآجونس باش ينحيلو الفراي.

template_id=3  (n=56)
  ex: Carte edinar mte3i tbloquet f dabouza w ma 3tatnich lflous. 7achti b déblocage express brabi rani 7asel.

template_id=20  (n=56)
  ex: الحريف يطلب clore son compte épargne و تحويل السولد (solde) لـ compte chèque متاعو. Il n'est plus intéressé par le taux d'intérêt actuel.

template_id=15  (n=53)
  ex: Reçu mail mta3 el client y7eb ya3mel rdv m3a el conseiller mta3o bech ychouf simulation de crédit immobilier. Klmto direct w fixit m3ah rdv pour mardi sba7.



### 3.2 Split Train / Validation / Test stratifié par template (SentimentLabel + ChurnRisk)

Un même `template_id` ne peut jamais se retrouver dans deux splits différents, **sauf exception
documentée** pour les templates dont la classe majoritaire (`SentimentLabel` OU `ChurnRisk`) est trop
rare (moins de 3 templates au total) pour être répartie sur 3 groupes sans qu'un split se retrouve vide :

- **Templates "normaux"** (classes majoritaires ≥ 3 templates pour les deux variables) : split par
  template via `MultilabelStratifiedShuffleSplit`, qui équilibre **simultanément** `SentimentLabel` et
  `ChurnRisk` entre Train / Validation / Test (garantie stricte, aucune fuite possible).
- **Templates "rares"** (une classe majoritaire, Sentiment ou Churn, avec < 3 templates au total) :
  split au niveau des lignes pour ces templates uniquement, stratifié sur la combinaison
  (Sentiment, Churn) quand c'est possible. Une fuite limitée à ces lignes est alors possible, mais
  c'est le seul moyen d'avoir des exemples de ces classes dans les 3 splits. La cellule 3.3 liste
  explicitement les templates concernés.


In [5]:
!pip install iterative-stratification --quiet

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

rng = np.random.RandomState(CONFIG["RANDOM_STATE"])

MIN_TEMPLATES_POUR_SPLIT_GROUPE = 3

# Etape 1 : classe majoritaire de chaque variable cible, par template_id
sentiment_par_template = (
    df.groupby("template_id")["SentimentLabel"].agg(lambda s: s.value_counts().idxmax())
)
churn_par_template = (
    df.groupby("template_id")["ChurnRisk"].agg(lambda s: s.value_counts().idxmax())
)
profil_template = pd.DataFrame({
    "SentimentLabel": sentiment_par_template,
    "ChurnRisk": churn_par_template,
})

# Etape 2 : reperer les classes trop rares (< 3 templates) sur CHACUNE des 2 variables
n_templates_par_classe_sentiment = profil_template["SentimentLabel"].value_counts()
n_templates_par_classe_churn = profil_template["ChurnRisk"].value_counts()
classes_rares_sentiment = set(n_templates_par_classe_sentiment[n_templates_par_classe_sentiment < MIN_TEMPLATES_POUR_SPLIT_GROUPE].index)
classes_rares_churn = set(n_templates_par_classe_churn[n_templates_par_classe_churn < MIN_TEMPLATES_POUR_SPLIT_GROUPE].index)

# Un template est "a risque" si SA classe majoritaire (Sentiment OU Churn) est rare
templates_fuite_documentee = set(
    profil_template.index[
        profil_template["SentimentLabel"].isin(classes_rares_sentiment)
        | profil_template["ChurnRisk"].isin(classes_rares_churn)
    ]
)
templates_normaux = profil_template.index.difference(templates_fuite_documentee)

if templates_fuite_documentee:
    print("Templates trop rares pour un split par groupe (fallback ligne-par-ligne) :")
    for tid in sorted(templates_fuite_documentee):
        print(f"  - template_id={tid} -> Sentiment={profil_template.loc[tid, 'SentimentLabel']}, "
              f"Churn={profil_template.loc[tid, 'ChurnRisk']}")
    print()

# --- Etape 3a : split PAR TEMPLATE des templates "normaux", stratifie sur les 2 variables a la fois ---
train_templates, val_templates, test_templates = [], [], []

if len(templates_normaux) > 0:
    profil_normal = profil_template.loc[templates_normaux]
    Y = pd.get_dummies(profil_normal).values.astype(int)
    X = np.zeros((len(templates_normaux), 1))
    ids_normaux = np.array(templates_normaux)

    # Train vs Reste (Validation + Test)
    msss1 = MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=CONFIG["VAL_SIZE"] + CONFIG["TEST_SIZE"], random_state=CONFIG["RANDOM_STATE"]
    )
    idx_train, idx_temp = next(msss1.split(X, Y))
    train_templates = list(ids_normaux[idx_train])

    # Validation vs Test, au sein du Reste
    part_test_dans_reste = CONFIG["TEST_SIZE"] / (CONFIG["VAL_SIZE"] + CONFIG["TEST_SIZE"])
    msss2 = MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=part_test_dans_reste, random_state=CONFIG["RANDOM_STATE"]
    )
    idx_val, idx_test = next(msss2.split(X[idx_temp], Y[idx_temp]))
    ids_temp = ids_normaux[idx_temp]
    val_templates = list(ids_temp[idx_val])
    test_templates = list(ids_temp[idx_test])

# --- Etape 3b : split PAR LIGNE des templates "rares", stratifie sur (Sentiment, Churn) si possible ---
lignes_train_fallback, lignes_val_fallback, lignes_test_fallback = [], [], []

if templates_fuite_documentee:
    rows = df.index[df["template_id"].isin(templates_fuite_documentee)].tolist()
    cle_stratif = df.loc[rows, "SentimentLabel"].astype(str) + "_" + df.loc[rows, "ChurnRisk"].astype(str)

    def split_avec_repli(rows, cle, test_size, random_state):
        # Essaie un split stratifie sur (Sentiment, Churn), puis sur Sentiment
        # seul, puis un split purement aleatoire si aucune stratification
        # n'est possible (classes trop rares meme au niveau des lignes).
        for strate in (cle, cle.str.split("_").str[0], None):
            try:
                return train_test_split(rows, test_size=test_size, random_state=random_state, stratify=strate)
            except ValueError:
                continue
        return train_test_split(rows, test_size=test_size, random_state=random_state)

    rows_train, rows_temp = split_avec_repli(
        rows, cle_stratif, CONFIG["VAL_SIZE"] + CONFIG["TEST_SIZE"], CONFIG["RANDOM_STATE"]
    )
    cle_temp = df.loc[rows_temp, "SentimentLabel"].astype(str) + "_" + df.loc[rows_temp, "ChurnRisk"].astype(str)
    part_test_dans_temp = CONFIG["TEST_SIZE"] / (CONFIG["VAL_SIZE"] + CONFIG["TEST_SIZE"])
    rows_val, rows_test = split_avec_repli(
        rows_temp, cle_temp, part_test_dans_temp, CONFIG["RANDOM_STATE"]
    )

    lignes_train_fallback, lignes_val_fallback, lignes_test_fallback = rows_train, rows_val, rows_test

# --- Assemblage final ---
df_train = pd.concat([
    df[df["template_id"].isin(train_templates)],
    df.loc[lignes_train_fallback],
]).reset_index(drop=True)

df_val = pd.concat([
    df[df["template_id"].isin(val_templates)],
    df.loc[lignes_val_fallback],
]).reset_index(drop=True)

df_test = pd.concat([
    df[df["template_id"].isin(test_templates)],
    df.loc[lignes_test_fallback],
]).reset_index(drop=True)

total = len(df_train) + len(df_val) + len(df_test)
print(f"Train      : {len(df_train):4d} lignes ({len(df_train) / total:.1%})")
print(f"Validation : {len(df_val):4d} lignes ({len(df_val) / total:.1%})")
print(f"Test       : {len(df_test):4d} lignes ({len(df_test) / total:.1%})")
print(f"Total      : {total} / {len(df)}")


Templates trop rares pour un split par groupe (fallback ligne-par-ligne) :
  - template_id=6 -> Sentiment=Neutre, Churn=Non applicable
  - template_id=10 -> Sentiment=Neutre, Churn=Non applicable
  - template_id=14 -> Sentiment=Positif, Churn=Non applicable

Train      :  789 lignes (78.9%)
Validation :  124 lignes (12.4%)
Test       :   87 lignes (8.7%)
Total      : 1000 / 1000


### 3.3 Garde-fou anti-fuite

Vérification stricte : aucun `template_id` ne doit apparaître dans plus d'un split, **à l'exception**
des templates listés dans `templates_fuite_documentee` (classes trop rares pour être réparties sans
fuite -- voir 3.2). Pour tout autre template, le notebook s'arrête (AssertionError) plutôt que de
laisser une fuite passer silencieusement.

Cette vérification est encapsulée dans une fonction réutilisable `verifier_anti_fuite()` car on la
relance après toute modification manuelle des splits (voir section 3.6).


In [6]:
def verifier_anti_fuite(df_train, df_val, df_test, templates_fuite_documentee):
    tid_train = set(df_train["template_id"])
    tid_val = set(df_val["template_id"])
    tid_test = set(df_test["template_id"])

    fuite_train_val = (tid_train & tid_val) - templates_fuite_documentee
    fuite_train_test = (tid_train & tid_test) - templates_fuite_documentee
    fuite_val_test = (tid_val & tid_test) - templates_fuite_documentee

    assert not fuite_train_val, f"Fuite non documentée détectée entre Train et Validation : {fuite_train_val}"
    assert not fuite_train_test, f"Fuite non documentée détectée entre Train et Test : {fuite_train_test}"
    assert not fuite_val_test, f"Fuite non documentée détectée entre Validation et Test : {fuite_val_test}"

    print("OK : aucune fuite non documentée entre Train / Validation / Test.")
    print(f"Templates uniques -> Train: {len(tid_train)} | Validation: {len(tid_val)} | Test: {len(tid_test)}")
    if templates_fuite_documentee:
        n_lignes_fuite = df["template_id"].isin(templates_fuite_documentee).sum()
        print(f"Fuite documentée (classes trop rares) : {len(templates_fuite_documentee)} template(s), "
              f"{n_lignes_fuite} lignes concernées au total.")


verifier_anti_fuite(df_train, df_val, df_test, templates_fuite_documentee)


OK : aucune fuite non documentée entre Train / Validation / Test.
Templates uniques -> Train: 18 | Validation: 5 | Test: 5
Fuite documentée (classes trop rares) : 3 template(s), 119 lignes concernées au total.


### 3.4 Vérification de l'équilibre des classes entre splits

Le split par `IterativeStratification` équilibre les classes autant que possible, mais avec seulement 22
templates, certaines classes minoritaires (ex. `Positif`) restent proches de la limite. On vérifie ici que
chaque split contient bien au moins un exemple de chaque classe, pour `SentimentLabel` et `ChurnRisk`.


In [7]:
print("=" * 70)
print("SentimentLabel par split")
print("=" * 70)
for nom, sous_df in [("Train", df_train), ("Validation", df_val), ("Test", df_test)]:
    print(f"\n{nom} ({len(sous_df)} lignes) :")
    print(sous_df["SentimentLabel"].value_counts())
    print(sous_df["SentimentLabel"].value_counts(normalize=True).round(3))


SentimentLabel par split

Train (789 lignes) :
SentimentLabel
Négatif    694
Neutre      57
Positif     38
Name: count, dtype: int64
SentimentLabel
Négatif    0.880
Neutre     0.072
Positif    0.048
Name: proportion, dtype: float64

Validation (124 lignes) :
SentimentLabel
Négatif    112
Neutre       7
Positif      5
Name: count, dtype: int64
SentimentLabel
Négatif    0.903
Neutre     0.056
Positif    0.040
Name: proportion, dtype: float64

Test (87 lignes) :
SentimentLabel
Négatif    75
Neutre      7
Positif     5
Name: count, dtype: int64
SentimentLabel
Négatif    0.862
Neutre     0.080
Positif    0.057
Name: proportion, dtype: float64


In [8]:
print("=" * 70)
print("ChurnRisk par split")
print("=" * 70)
for nom, sous_df in [("Train", df_train), ("Validation", df_val), ("Test", df_test)]:
    print(f"\n{nom} ({len(sous_df)} lignes) :")
    print(sous_df["ChurnRisk"].value_counts())
    print(sous_df["ChurnRisk"].value_counts(normalize=True).round(3))


ChurnRisk par split

Train (789 lignes) :
ChurnRisk
Faible            363
Moyen             201
Élevé             130
Non applicable     95
Name: count, dtype: int64
ChurnRisk
Faible            0.460
Moyen             0.255
Élevé             0.165
Non applicable    0.120
Name: proportion, dtype: float64

Validation (124 lignes) :
ChurnRisk
Élevé             63
Faible            49
Non applicable    12
Name: count, dtype: int64
ChurnRisk
Élevé             0.508
Faible            0.395
Non applicable    0.097
Name: proportion, dtype: float64

Test (87 lignes) :
ChurnRisk
Moyen             39
Faible            36
Non applicable    12
Name: count, dtype: int64
ChurnRisk
Moyen             0.448
Faible            0.414
Non applicable    0.138
Name: proportion, dtype: float64


In [9]:
# Alerte explicite si une classe de SentimentLabel OU de ChurnRisk est
# totalement absente d'un split -- a documenter dans le rapport plutot
# qu'a decouvrir plus tard lors du fine-tuning.
for colonne in ["SentimentLabel", "ChurnRisk"]:
    toutes_classes = set(df[colonne].unique())
    for nom, sous_df in [("Train", df_train), ("Validation", df_val), ("Test", df_test)]:
        classes_absentes = toutes_classes - set(sous_df[colonne].unique())
        if classes_absentes:
            print(f"ATTENTION : {nom} ne contient aucun exemple de {classes_absentes} pour {colonne}")
        else:
            print(f"OK : {nom} contient au moins un exemple de chaque classe {colonne}.")
    print()


OK : Train contient au moins un exemple de chaque classe SentimentLabel.
OK : Validation contient au moins un exemple de chaque classe SentimentLabel.
OK : Test contient au moins un exemple de chaque classe SentimentLabel.

OK : Train contient au moins un exemple de chaque classe ChurnRisk.
ATTENTION : Validation ne contient aucun exemple de {'Moyen'} pour ChurnRisk
ATTENTION : Test ne contient aucun exemple de {'Élevé'} pour ChurnRisk



### 3.5 Vérification ciblée : ChurnRisk sur le sous-ensemble Sentiment = Négatif

Le modèle de churn (notebook 03) n'est entraîné et évalué **que sur les enregistrements de sentiment
négatif**. L'alerte de la section 3.4 peut donc être trompeuse : une classe `ChurnRisk` absente d'un
split sur l'ensemble complet n'est un problème que si elle est aussi absente sur le sous-ensemble négatif.
On vérifie donc spécifiquement la présence de `Élevé` et `Moyen` (les deux classes utiles pour le modèle
churn) dans chaque split, restreint aux lignes `SentimentLabel == "Négatif"`.


In [10]:
def verifier_churn_sur_negatif(df_train, df_val, df_test):
    print("=" * 70)
    print("ChurnRisk par split (sous-ensemble Sentiment = Négatif uniquement)")
    print("=" * 70)
    classes_churn_utiles = {"Élevé", "Moyen"}
    ok = True
    for nom, sous_df in [("Train", df_train), ("Validation", df_val), ("Test", df_test)]:
        sous_df_neg = sous_df[sous_df["SentimentLabel"] == "Négatif"]
        print(f"\n{nom} ({len(sous_df_neg)} lignes négatives) :")
        print(sous_df_neg["ChurnRisk"].value_counts())
        classes_absentes = classes_churn_utiles - set(sous_df_neg["ChurnRisk"].unique())
        if classes_absentes:
            print(f"ATTENTION : {nom} (négatif) ne contient aucun exemple de {classes_absentes}")
            ok = False
        else:
            print(f"OK : {nom} (négatif) contient Élevé et Moyen.")
    return ok


churn_ok = verifier_churn_sur_negatif(df_train, df_val, df_test)


ChurnRisk par split (sous-ensemble Sentiment = Négatif uniquement)

Train (694 lignes négatives) :
ChurnRisk
Faible    363
Moyen     201
Élevé     130
Name: count, dtype: int64
OK : Train (négatif) contient Élevé et Moyen.

Validation (112 lignes négatives) :
ChurnRisk
Élevé     63
Faible    49
Name: count, dtype: int64
ATTENTION : Validation (négatif) ne contient aucun exemple de {'Moyen'}

Test (75 lignes négatives) :
ChurnRisk
Moyen     39
Faible    36
Name: count, dtype: int64
ATTENTION : Test (négatif) ne contient aucun exemple de {'Élevé'}


### 3.6 Réassignation manuelle ciblée (si nécessaire)

Si la vérification 3.5 signale un trou, on déplace **le plus petit template éligible** (Sentiment négatif,
classe ChurnRisk manquante, actuellement en Train) vers le split concerné, plutôt que le premier trouvé,
pour limiter l'impact sur les proportions Train/Validation/Test. Le déplacement se fait au niveau du
template entier (jamais d'une ligne isolée) pour ne pas introduire de fuite.

Cette cellule ne fait rien si `churn_ok` est déjà `True`.


In [11]:
def trouver_plus_petit_candidat(classe_churn, exclure, train_templates):
    candidats = profil_template.index[
        (profil_template["ChurnRisk"] == classe_churn)
        & (profil_template["SentimentLabel"] == "Négatif")
        & profil_template.index.isin(train_templates)
        & ~profil_template.index.isin(exclure)
    ]
    if len(candidats) == 0:
        return None
    tailles = df[df["template_id"].isin(candidats)]["template_id"].value_counts()
    return tailles.idxmin()  # le plus petit template éligible, pas le premier trouvé


def deplacer_template(tid, source, dest):
    source.remove(tid)
    dest.append(tid)


if not churn_ok:
    if not (profil_template.loc[list(test_templates), "ChurnRisk"] == "Élevé").any():
        cand = trouver_plus_petit_candidat("Élevé", test_templates, train_templates)
        if cand is not None:
            deplacer_template(cand, train_templates, test_templates)
            print(f"Template {cand} (Élevé) déplacé de Train vers Test.")
        else:
            print("Aucun template Négatif+Élevé disponible en Train à déplacer.")

    if not (profil_template.loc[list(val_templates), "ChurnRisk"] == "Moyen").any():
        cand = trouver_plus_petit_candidat("Moyen", val_templates, train_templates)
        if cand is not None:
            deplacer_template(cand, train_templates, val_templates)
            print(f"Template {cand} (Moyen) déplacé de Train vers Validation.")
        else:
            print("Aucun template Négatif+Moyen disponible en Train à déplacer.")

    # Reconstruction des 3 splits avec les listes de templates mises à jour
    df_train = pd.concat([
        df[df["template_id"].isin(train_templates)],
        df.loc[lignes_train_fallback],
    ]).reset_index(drop=True)

    df_val = pd.concat([
        df[df["template_id"].isin(val_templates)],
        df.loc[lignes_val_fallback],
    ]).reset_index(drop=True)

    df_test = pd.concat([
        df[df["template_id"].isin(test_templates)],
        df.loc[lignes_test_fallback],
    ]).reset_index(drop=True)

    total = len(df_train) + len(df_val) + len(df_test)
    print(f"\nNouvelles proportions -> Train: {len(df_train)/total:.1%} | "
          f"Validation: {len(df_val)/total:.1%} | Test: {len(df_test)/total:.1%}")

    # On revérifie systématiquement après tout déplacement manuel :
    # 1) qu'aucune fuite non documentée n'a été introduite
    # 2) que le trou est bien comblé
    verifier_anti_fuite(df_train, df_val, df_test, templates_fuite_documentee)
    churn_ok = verifier_churn_sur_negatif(df_train, df_val, df_test)
    assert churn_ok, "Le trou ChurnRisk (sous-ensemble négatif) persiste après réassignation manuelle."
else:
    print("Rien à faire : churn_ok déjà True, aucune réassignation nécessaire.")


Template 9 (Élevé) déplacé de Train vers Test.
Template 5 (Moyen) déplacé de Train vers Validation.

Nouvelles proportions -> Train: 71.1% | Validation: 16.6% | Test: 12.3%
OK : aucune fuite non documentée entre Train / Validation / Test.
Templates uniques -> Train: 16 | Validation: 6 | Test: 6
Fuite documentée (classes trop rares) : 3 template(s), 119 lignes concernées au total.
ChurnRisk par split (sous-ensemble Sentiment = Négatif uniquement)

Train (616 lignes négatives) :
ChurnRisk
Faible    363
Moyen     159
Élevé      94
Name: count, dtype: int64
OK : Train (négatif) contient Élevé et Moyen.

Validation (154 lignes négatives) :
ChurnRisk
Élevé     63
Faible    49
Moyen     42
Name: count, dtype: int64
OK : Validation (négatif) contient Élevé et Moyen.

Test (111 lignes négatives) :
ChurnRisk
Moyen     39
Faible    36
Élevé     36
Name: count, dtype: int64
OK : Test (négatif) contient Élevé et Moyen.


### 4. Sauvegarde

On sauvegarde les trois splits séparément, ainsi qu'une colonne `split` sur le corpus complet pour
faciliter la traçabilité (permet de retrouver rapidement à quel split appartient une ligne donnée).


In [12]:
from pathlib import Path

dossier_sortie = Path("/content/drive/MyDrive/data/processed")
dossier_sortie.mkdir(parents=True, exist_ok=True)

chemin_train = dossier_sortie / "comptes_rendus_train.csv"
chemin_val = dossier_sortie / "comptes_rendus_validation.csv"
chemin_test = dossier_sortie / "comptes_rendus_test.csv"

df_train.to_csv(chemin_train, index=False)
df_val.to_csv(chemin_val, index=False)
df_test.to_csv(chemin_test, index=False)

print("Fichiers sauvegardés sur Drive :")
print(" -", chemin_train)
print(" -", chemin_val)
print(" -", chemin_test)


Fichiers sauvegardés sur Drive :
 - /content/drive/MyDrive/data/processed/comptes_rendus_train.csv
 - /content/drive/MyDrive/data/processed/comptes_rendus_validation.csv
 - /content/drive/MyDrive/data/processed/comptes_rendus_test.csv
